# 03 — BQ1: Where and When Do Students Drop Out?

> **Notebook 03 of 7** | Learning Retention Analytics  
> First business question analysis: temporal patterns of student withdrawal across courses.

## Purpose and Scope

This notebook answers **BQ1: Where and when do students drop out?** — the first of five business questions driving this project.

**What this notebook covers:**
- Scale of the dropout problem: how many students withdraw, and from which courses
- Cumulative dropout curves — survival-style analysis showing when students leave
- Cliff detection — identifying critical moments of mass withdrawal
- Normalized timeline — comparing dropout timing across courses of different lengths
- Pre-course withdrawals — students who leave before the course even starts
- Demographic segmentation of dropout timing
- Course design characteristics and their relationship to dropout patterns

**What comes next:**
- **Notebook 04** (`04_bq2_early_signals.ipynb`): which early behavioral signals predict drop-out? (BQ2)

**Connection to other business questions:**  
BQ1 establishes *when* students leave. BQ2 (Notebook 04) will identify behavioral signals that *predict* departure. Together, they define both the intervention window and the triggers for early warning systems.

> **Methodological transferability:** The survival-style dropout analysis, cliff detection, and normalized timeline comparison used here are directly applicable to SaaS churn analysis (when do subscribers cancel?), subscription cohort decay (which cohorts retain better?), and app engagement drop-off (when do users stop returning?). The domain is education; the analytical framework is retention analytics.

## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [Dropout Overview — Scale and Rate](#2.-Dropout-Overview-—-Scale-and-Rate)
3. [Cumulative Dropout Curves](#3.-Cumulative-Dropout-Curves)
4. [Identifying Dropout Cliffs](#4.-Identifying-Dropout-Cliffs)
5. [Normalized Timeline — Percentage Through Course](#5.-Normalized-Timeline-—-Percentage-Through-Course)
6. [Pre-Course Withdrawals](#6.-Pre-Course-Withdrawals)
7. [Dropout Timing by Demographics](#7.-Dropout-Timing-by-Demographics)
8. [Course Design and Dropout Patterns](#8.-Course-Design-and-Dropout-Patterns)
9. [Key Takeaways and Next Steps](#9.-Key-Takeaways-and-Next-Steps)

---

**Prerequisites:**
- The ETL pipeline must have been run: `python -m run_pipeline`
- The DuckDB database at `data/db/oulad.duckdb` must contain all 5 analytical views

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K students, 7 courses, complete behavioral clickstream. License: CC-BY 4.0.

## 1. Environment Setup

We configure imports, visualization defaults, and reusable helper functions.

**Technical notes for readers:**
- Notebooks live in `notebooks/` but project modules are in `src/` at the project root. We add the project root to `sys.path` so that `from src.config import ...` works.
- All database queries go through `src.db.connection.execute_query()` — the project's DB abstraction layer (ADR-003).
- BQ1's primary SQL query lives in `sql/queries/q_bq1_dropout_curves.sql` and is loaded at runtime from disk — no SQL is hardcoded in Python (ADR-003).
- Figures are saved to `reports/figures/` at 150 DPI. Since `nbstripout` removes notebook outputs before commit, the saved PNGs are the persistent visual record.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query

# --- Configuration ---
# Suppress noisy warnings in notebook output; errors still surface
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
# Consistent style across all project notebooks
sns.set_theme(style='whitegrid', font_scale=1.1)

# Semantic color palette: one color per outcome category
PALETTE_OUTCOME = {
    'Pass': '#4C72B0',        # blue — neutral positive
    'Distinction': '#55A868', # green — strong positive
    'Fail': '#C44E52',        # red — negative
    'Withdrawn': '#8172B3',   # purple — departed (distinct from failed)
}
# Outcome label constants — single source of truth for binary labels
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
PALETTE_SEQUENTIAL = 'YlOrRd'

# Course-level palette: one color per module for consistent identification
# across all dropout curve and comparison charts. The 7 OULAD modules use
# tab10 for maximum visual distinction between course lines.
_MODULE_ORDER = ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
_TAB10 = plt.cm.tab10.colors
PALETTE_COURSE = {m: _TAB10[i] for i, m in enumerate(_MODULE_ORDER)}

# Shared axis labels — avoids cross-cell string literal duplication
LABEL_DROPOUT_DAY = 'Dropout day (relative to course start)'
LABEL_CUMULATIVE_DROPOUT = 'Cumulative dropout rate (%)'
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_MEDIAN_DROPOUT_DAY = 'Median dropout day'
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

# Ensure figures output directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ1 query from SQL file ---
# The primary query lives in sql/queries/ as a standalone file (ADR-003).
# Loading it once here avoids re-reading the file in multiple cells.
_bq1_sql_path = QUERIES_DIR / 'q_bq1_dropout_curves.sql'
BQ1_SQL = _bq1_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ1 query from: {_bq1_sql_path.name} ({len(BQ1_SQL):,} chars)')

# --- Prerequisite check ---
# Verify the database is populated before proceeding.
try:
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_dropout = execute_query('SELECT COUNT(*) AS n FROM v_dropout_timing')
    _n_student = _check_student['n'].iloc[0]
    _n_dropout = _check_dropout['n'].iloc[0]
    if _n_student == 0 or _n_dropout == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
    print(f'  v_dropout_timing:    {_n_dropout:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Dropout Overview — Scale and Rate

Before examining *when* students drop out, we establish **how many** drop out and from which courses. This section provides the baseline numbers that contextualize all subsequent analyses.

**Key distinction:** In the OULAD dataset, "withdrawn" means the student explicitly unregistered from the course (`date_unregistration IS NOT NULL`). Students who stayed enrolled but failed are classified as "Fail", not "Withdrawn". This notebook focuses on **explicit withdrawals** — the students who actively chose to leave.

In [ ]:
# --- Dropout headline numbers ---
# Count explicit withdrawals (withdrew_explicit = 1) vs total enrollments
# per module to establish the scale of the dropout problem.
df_overview = execute_query('''
    SELECT
        code_module,
        COUNT(*) AS n_enrolled,
        SUM(withdrew_explicit) AS n_withdrew,
        ROUND(100.0 * SUM(withdrew_explicit) / COUNT(*), 1) AS withdrawal_rate_pct
    FROM v_student_enriched
    GROUP BY code_module
    ORDER BY withdrawal_rate_pct DESC
''')

# Overall numbers
total_enrolled = df_overview['n_enrolled'].sum()
total_withdrew = df_overview['n_withdrew'].sum()
overall_rate = 100.0 * total_withdrew / total_enrolled

print('=== Dropout Overview ===\n')
print(f'  Total enrollments:     {total_enrolled:>8,}')
print(f'  Explicit withdrawals:  {total_withdrew:>8,} ({overall_rate:.1f}%)')
print(f'  Stayed enrolled:       {total_enrolled - total_withdrew:>8,} ({100 - overall_rate:.1f}%)')
print('\n=== Withdrawal Rate by Module ===\n')
print(df_overview.to_string(index=False))

# --- Horizontal bar chart: withdrawal count + rate per module ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

modules = df_overview['code_module']
colors = [PALETTE_COURSE[m] for m in modules]

# Left: absolute withdrawal count
ax1.barh(modules, df_overview['n_withdrew'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_overview.iterrows()):
    ax1.text(
        row['n_withdrew'] + ax1.get_xlim()[1] * 0.01, i,
        f"{int(row['n_withdrew']):,}",
        va='center', fontsize=9, color='#333333',
    )
ax1.set_xlabel('Number of withdrawals')
ax1.set_title('Withdrawal Count by Module')
ax1.invert_yaxis()
sns.despine(ax=ax1)

# Right: withdrawal rate percentage
ax2.barh(modules, df_overview['withdrawal_rate_pct'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_overview.iterrows()):
    ax2.text(
        row['withdrawal_rate_pct'] + 0.5, i,
        f"{row['withdrawal_rate_pct']:.1f}%",
        va='center', fontsize=9, color='#333333',
    )
# Reference line: overall withdrawal rate across all modules
ax2.axvline(x=overall_rate, color='gray', linestyle='--', linewidth=1,
            label=f'Overall: {overall_rate:.1f}%')
ax2.set_xlabel('Withdrawal rate (%)')
ax2.set_title('Withdrawal Rate by Module')
ax2.legend(loc='lower right')
ax2.invert_yaxis()
sns.despine(ax=ax2)

fig.suptitle('Dropout Overview by Module', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_overview_by_course')
plt.show()

> **Key finding:** Roughly one-third of all enrollments end in explicit withdrawal. The withdrawal rate varies across modules — some courses lose a significantly larger share of students than others.
>
> This variation is important: BQ4 (Notebook 06) will investigate whether course design features (assessment density, resource count, duration) explain these differences. For now, we note that the problem is substantial and module-dependent.

## 3. Cumulative Dropout Curves

This is the **core BQ1 visualization** — a survival-style analysis showing how dropout accumulates over time.

The query `q_bq1_dropout_curves.sql` computes cumulative dropout counts and percentages per course-presentation using window functions (`SUM OVER ... ROWS BETWEEN UNBOUNDED PRECEDING`). Each point on the curve represents the percentage of the original cohort that has withdrawn by that day.

**Why step charts?** Dropout is a discrete event — students withdraw on specific days, not continuously. A step chart (rather than a smoothed line) preserves this reality and makes cliff events (sudden spikes) visible.

In [ ]:
# --- Execute BQ1 query ---
# The SQL file was loaded in the setup cell. Each row represents a day
# when at least one dropout occurred, with running totals and percentages.
df_curves = execute_query(BQ1_SQL)

print(f'BQ1 result: {len(df_curves):,} rows')
print(f'Courses: {df_curves["code_module"].nunique()} modules, '
      f'{df_curves[["code_module", "code_presentation"]].drop_duplicates().shape[0]} presentations')

# --- Overlaid cumulative dropout curves ---
# One line per course-presentation, colored by module. This shows the
# full landscape: how different cohorts decay over time.
fig, ax = plt.subplots(figsize=(12, 7))

for (module, pres), group in df_curves.groupby(['code_module', 'code_presentation']):
    ax.step(
        group['dropout_day'], group['cumulative_dropout_rate_pct'],
        where='post', color=PALETTE_COURSE[module], alpha=0.7,
        linewidth=1.5, label=None,
    )

# Add one legend entry per module (not per presentation)
for module in _MODULE_ORDER:
    if module in df_curves['code_module'].values:
        ax.plot([], [], color=PALETTE_COURSE[module], linewidth=2, label=module)

ax.set_xlabel(LABEL_DROPOUT_DAY)
ax.set_ylabel(LABEL_CUMULATIVE_DROPOUT)
ax.set_title('Cumulative Dropout Curves — All Course-Presentations')
ax.legend(title='Module', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, None)
sns.despine()
fig.tight_layout()
save_fig(fig, '03_dropout_curves_overlaid')
plt.show()

In [ ]:
# --- Faceted small multiples: one panel per module ---
# Separating modules into individual panels avoids the visual clutter of
# the overlaid view and lets the reader focus on each course's trajectory.
modules_present = sorted(df_curves['code_module'].unique())
n_modules = len(modules_present)
n_cols = 4
n_rows = (n_modules + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), sharey=True)
axes_flat = axes.flatten()

for i, module in enumerate(modules_present):
    ax = axes_flat[i]
    module_data = df_curves[df_curves['code_module'] == module]

    for pres, group in module_data.groupby('code_presentation'):
        ax.step(
            group['dropout_day'], group['cumulative_dropout_rate_pct'],
            where='post', color=PALETTE_COURSE[module], alpha=0.7,
            linewidth=1.5, label=pres,
        )

    ax.set_title(module, fontsize=12, fontweight='bold')
    ax.set_xlabel(LABEL_DROPOUT_DAY)
    if i % n_cols == 0:
        ax.set_ylabel(LABEL_CUMULATIVE_DROPOUT)
    ax.legend(fontsize=7, loc='lower right')
    ax.set_ylim(0, None)
    sns.despine(ax=ax)

# Hide unused subplots
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Cumulative Dropout Curves by Module', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_curves_faceted')
plt.show()

> **Interpretation:**
> - The dropout curves reveal **different temporal profiles** across modules. Some courses experience steady, gradual attrition; others show sharp drops at specific points.
> - Within the same module, different presentations (cohorts) follow broadly similar trajectories, suggesting that the dropout pattern is more a function of course design than cohort composition.
> - Most dropout curves show an **initial steep segment** (early departures) followed by a slower, more gradual decline. This pattern is characteristic of survival curves across many domains.
>
> **SaaS parallel:** These curves are directly analogous to *cohort retention curves* in subscription businesses. The shape of the curve (convex vs. concave, steep vs. gradual) reveals whether churn is concentrated in onboarding (activation problem) or distributed over time (value-delivery problem).

## 4. Identifying Dropout Cliffs

Not all days are equal. Some days see **disproportionately large numbers of withdrawals** — "dropout cliffs" that may correspond to specific course events (assessment deadlines, feedback release, registration cutoffs).

We identify cliffs by looking at the daily dropout count from the BQ1 query results and flagging days where the count exceeds the 95th percentile for that course. These are days when something triggered an unusual number of departures.

In [ ]:
# --- Cliff detection: days with unusually high dropout counts ---
# For each course-presentation, flag days where daily dropout count
# exceeds the 95th percentile. These outlier days represent sudden
# mass departures — likely tied to course milestones.
df_daily_dropouts = df_curves[[
    'code_module', 'code_presentation', 'dropout_day', 'n_dropouts'
]].copy()

# 95th percentile threshold per course-presentation
thresholds = (
    df_daily_dropouts
    .groupby(['code_module', 'code_presentation'])['n_dropouts']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'n_dropouts': 'p95_threshold'})
)

df_cliffs = df_daily_dropouts.merge(thresholds, on=['code_module', 'code_presentation'])
df_cliffs = df_cliffs[df_cliffs['n_dropouts'] >= df_cliffs['p95_threshold']]

# Aggregate across presentations: which module-day combos are cliffs most often?
df_cliff_summary = (
    df_cliffs
    .groupby(['code_module', 'dropout_day'])
    .agg(
        total_dropouts=('n_dropouts', 'sum'),
        n_presentations=('code_presentation', 'nunique'),
    )
    .reset_index()
    .sort_values('total_dropouts', ascending=False)
)

print('=== Top Cliff Events (days with >= p95 dropouts) ===\n')
print(df_cliff_summary.head(15).to_string(index=False))

# --- Visualization: top cliff events across all modules ---
df_cliff_top = df_cliff_summary.head(20)
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_COURSE.get(m, '#999999') for m in df_cliff_top['code_module']]
ax.barh(
    range(len(df_cliff_top)),
    df_cliff_top['total_dropouts'],
    color=colors, edgecolor='white',
)

# Label each bar with module + day for identification
labels = [
    f"{row['code_module']} — day {int(row['dropout_day'])}"
    for _, row in df_cliff_top.iterrows()
]
ax.set_yticks(range(len(df_cliff_top)))
ax.set_yticklabels(labels)
ax.set_xlabel('Number of dropouts')
ax.set_title('Top Dropout Cliff Events (days exceeding 95th percentile)')
ax.invert_yaxis()
sns.despine()
fig.tight_layout()
save_fig(fig, '03_dropout_cliffs')
plt.show()

> **Interpretation:**
> - Cliff events are concentrated at **specific days** that likely correspond to course milestones: assessment deadlines, grade releases, or registration cutoff dates.
> - Some modules show cliffs earlier in the timeline (suggesting onboarding-phase attrition), while others show cliffs mid-course (suggesting assessment-driven attrition).
> - The consistency of cliff timing across presentations of the same module strengthens the hypothesis that these events are tied to **course design** rather than student-specific factors.
>
> **Actionable insight:** If cliff days align with known course events, the platform operator can deploy targeted interventions *before* those dates — e.g., reminder emails, support resources, or simplified re-enrollment pathways for students who missed assessment deadlines.

## 5. Normalized Timeline — Percentage Through Course

OULAD courses have different lengths (ranging from ~240 to ~270 days). Comparing dropout days in absolute terms mixes apples and oranges. To enable cross-course comparison, `v_dropout_timing` includes a `dropout_pct` field: the percentage of course duration elapsed when the student withdrew.

- 0% = withdrew at course start (day 0)
- 50% = withdrew halfway through the course
- 100% = withdrew at the scheduled end

Negative values represent **pre-course withdrawals** (students who unregistered before the course officially started). Values above 100% are possible if a student withdrew after the scheduled end date.

In [ ]:
# --- Normalized dropout timing distribution ---
# Using v_dropout_timing.dropout_pct to compare across courses.
# This DataFrame is reused in Section 6 (pre-course withdrawals).
df_dropout_timing = execute_query('SELECT * FROM v_dropout_timing')

# Separate in-course and pre-course withdrawals
df_in_course = df_dropout_timing[
    (df_dropout_timing['dropout_pct'] >= 0) & (df_dropout_timing['dropout_pct'] <= 100)
]
df_pre_course = df_dropout_timing[df_dropout_timing['dropout_pct'] < 0]

n_post = len(df_dropout_timing[df_dropout_timing['dropout_pct'] > 100])
print(f'Total withdrawals:        {len(df_dropout_timing):,}')
print(f'In-course (0-100%):       {len(df_in_course):,}')
print(f'Pre-course (< 0%):        {len(df_pre_course):,}')
print(f'Post-course (> 100%):     {n_post:,}')

# --- Histogram + KDE of normalized dropout timing ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left: overall histogram for in-course withdrawals
ax1.hist(
    df_in_course['dropout_pct'], bins=50, color='#8172B3',
    edgecolor='white', alpha=0.8,
)
ax1.set_xlabel('Percentage through course at withdrawal')
ax1.set_ylabel(LABEL_NUM_ENROLLMENTS)
ax1.set_title('Dropout Timing Distribution (normalized)')
ax1.axvline(x=50, color='gray', linestyle='--', linewidth=1, label='Midpoint')
ax1.legend()
sns.despine(ax=ax1)

# Right: KDE by module for cross-course comparison
for module in sorted(df_in_course['code_module'].unique()):
    subset = df_in_course[df_in_course['code_module'] == module]
    # KDE requires enough data points to be meaningful
    if len(subset) > 10:
        sns.kdeplot(
            subset['dropout_pct'], ax=ax2,
            color=PALETTE_COURSE[module], label=module, linewidth=1.5,
        )
ax2.set_xlabel('Percentage through course at withdrawal')
ax2.set_ylabel('Density')
ax2.set_title('Dropout Timing by Module (KDE, normalized)')
ax2.legend(title='Module')
sns.despine(ax=ax2)

fig.suptitle('When Do Students Drop Out? (normalized by course length)',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_normalized')
plt.show()

> **Interpretation:**
> - The normalized histogram reveals the **phases** of dropout: early attrition (0–20%), mid-course departure (around 30–50%), and late-stage withdrawal (beyond 60%).
> - The KDE overlay shows that different modules have **different dropout timing profiles**. Some courses see most dropout in the first quarter, while others have a more gradual or mid-course-heavy pattern.
> - The fact that a non-trivial share of withdrawals happen beyond 50% is notable: these students invested significant time before leaving. Late-stage dropout may be driven by assessment failure or academic difficulty, rather than initial disengagement.
>
> **SaaS parallel:** The normalized timeline is equivalent to *lifecycle stage analysis* in subscription businesses. Early churn (activation failure) requires different interventions than mid-lifecycle churn (value gap) or late churn (competitive displacement or life change).

## 6. Pre-Course Withdrawals

A surprising subset of students withdraw **before the course even starts** (`dropout_day < 0`). These are enrollments where the student registered and then unregistered before day 0.

Pre-course withdrawals represent a distinct phenomenon: the student never experienced any course content. This is pure **registration churn** — analogous to SaaS users who sign up during a trial and cancel before ever using the product.

In [ ]:
# --- Pre-course withdrawal analysis ---
# Students who withdrew before day 0 (negative dropout_day).
df_precourse = execute_query('''
    SELECT
        code_module,
        COUNT(*) AS n_precourse,
        ROUND(
            100.0 * COUNT(*)
            / SUM(COUNT(*)) OVER (),
            1
        ) AS pct_of_precourse
    FROM v_dropout_timing
    WHERE dropout_day < 0
    GROUP BY code_module
    ORDER BY n_precourse DESC
''')

# What share of all withdrawals are pre-course?
total_withdrew_all = len(df_dropout_timing)
total_precourse = df_precourse['n_precourse'].sum()
precourse_pct = 100.0 * total_precourse / total_withdrew_all

print('=== Pre-Course Withdrawals ===\n')
print(f'  Total withdrawals:      {total_withdrew_all:>8,}')
print(f'  Pre-course (day < 0):   {total_precourse:>8,} ({precourse_pct:.1f}% of all withdrawals)')
print('\n=== Pre-Course by Module ===\n')
print(df_precourse.to_string(index=False))

# --- Bar chart: pre-course withdrawals by module ---
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_COURSE.get(m, '#999999') for m in df_precourse['code_module']]
bars = ax.barh(
    df_precourse['code_module'], df_precourse['n_precourse'],
    color=colors, edgecolor='white',
)
for bar, (_, row) in zip(bars, df_precourse.iterrows()):
    ax.text(
        bar.get_width() + ax.get_xlim()[1] * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['n_precourse']):,} ({row['pct_of_precourse']:.1f}%)",
        va='center', fontsize=9, color='#333333',
    )
ax.set_xlabel('Number of pre-course withdrawals')
ax.set_title('Pre-Course Withdrawals by Module (withdrew before day 0)')
ax.invert_yaxis()
sns.despine()
fig.tight_layout()
save_fig(fig, '03_precourse_withdrawals')
plt.show()

> **Key finding:** A measurable share of all withdrawals happen before the course even begins. These students registered but never engaged with any content.
>
> - Pre-course withdrawal is a **registration quality** signal: the enrollment funnel converts users who are not yet committed.
> - The distribution across modules suggests that some courses may have better registration-to-start conversion than others.
>
> **Intervention opportunity:** Pre-course withdrawals could be reduced with better expectation-setting at registration time, welcome emails with concrete "first steps", or a simplified late-registration pathway. In SaaS terms, this is the "activation email" problem — the user signed up but needs a nudge to actually start.

## 7. Dropout Timing by Demographics

Does *when* students drop out vary by demographic group? `v_dropout_timing` includes demographic columns from `v_student_enriched`, allowing us to segment dropout timing by education level, age band, and deprivation index.

**Important distinction:** This section is **descriptive only** — we observe differences in median dropout day but do not perform statistical tests. Formal hypothesis testing on demographic associations belongs in Notebook 05 (BQ3: demographics vs. behavior).

In [ ]:
# --- Dropout timing by demographics ---
# Compute median dropout day for each demographic group.
# Using PERCENTILE_CONT for ANSI compliance.
# Only in-course withdrawals (dropout_day >= 0) to focus on the course experience.
df_demo_dropout = execute_query('''
    SELECT
        highest_education,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day,
        COUNT(*) AS n
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY highest_education
    HAVING COUNT(*) >= 30
    ORDER BY median_dropout_day
''')

df_age_dropout = execute_query('''
    SELECT
        age_band,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day,
        COUNT(*) AS n
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY age_band
    HAVING COUNT(*) >= 30
    ORDER BY median_dropout_day
''')

df_imd_dropout = execute_query('''
    SELECT
        COALESCE(CAST(imd_band AS VARCHAR), 'Unknown') AS imd_band,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day,
        COUNT(*) AS n
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY imd_band
    HAVING COUNT(*) >= 30
    ORDER BY median_dropout_day
''')

# --- 1x3 bar chart panel ---
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

# Left: by education level
ax1.barh(df_demo_dropout['highest_education'], df_demo_dropout['median_dropout_day'],
         color='#4C72B0', edgecolor='white')
for i, (_, row) in enumerate(df_demo_dropout.iterrows()):
    ax1.text(
        row['median_dropout_day'] + 1, i,
        f"day {int(row['median_dropout_day'])} (n={int(row['n']):,})",
        va='center', fontsize=8, color='#333333',
    )
ax1.set_xlabel(LABEL_MEDIAN_DROPOUT_DAY)
ax1.set_title('By Education Level')
ax1.invert_yaxis()
sns.despine(ax=ax1)

# Center: by age band
ax2.barh(df_age_dropout['age_band'], df_age_dropout['median_dropout_day'],
         color='#55A868', edgecolor='white')
for i, (_, row) in enumerate(df_age_dropout.iterrows()):
    ax2.text(
        row['median_dropout_day'] + 1, i,
        f"day {int(row['median_dropout_day'])} (n={int(row['n']):,})",
        va='center', fontsize=8, color='#333333',
    )
ax2.set_xlabel(LABEL_MEDIAN_DROPOUT_DAY)
ax2.set_title('By Age Band')
ax2.invert_yaxis()
sns.despine(ax=ax2)

# Right: by IMD band
ax3.barh(df_imd_dropout['imd_band'], df_imd_dropout['median_dropout_day'],
         color='#C44E52', edgecolor='white')
for i, (_, row) in enumerate(df_imd_dropout.iterrows()):
    ax3.text(
        row['median_dropout_day'] + 1, i,
        f"day {int(row['median_dropout_day'])} (n={int(row['n']):,})",
        va='center', fontsize=8, color='#333333',
    )
ax3.set_xlabel(LABEL_MEDIAN_DROPOUT_DAY)
ax3.set_title('By IMD Band')
ax3.invert_yaxis()
sns.despine(ax=ax3)

fig.suptitle('Median Dropout Day by Demographic Group (in-course withdrawals only)',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_by_demographics')
plt.show()

> **Interpretation:**
> - Median dropout day varies across demographic groups, but the differences are moderate compared to the overall variation in dropout timing.
> - **Education level** shows some differentiation: students with different educational backgrounds may drop out at different stages of the course lifecycle.
> - **Age band** and **IMD band** (socioeconomic deprivation index) show patterns that suggest demographic context influences not just *whether* students drop out, but *when*.
>
> **Caveat:** These are descriptive observations — no causal claims. The differences could be confounded by course choice, prior experience, or other factors. BQ3 (Notebook 05) will perform formal statistical tests comparing demographic and behavioral predictors.

## 8. Course Design and Dropout Patterns

Do course characteristics explain the variation in dropout rates across modules? `v_course_profile` provides design metrics for each course-presentation: duration, number of assessments, assessment density (assessments per 30 days), number of VLE resources, and activity type diversity.

This section looks for **visual correlations** between course design features and dropout behavior. A full course comparison is the subject of BQ4 (Notebook 06); here we establish the patterns.

In [ ]:
# --- Course profile data ---
df_course = execute_query('SELECT * FROM v_course_profile')

# --- Median dropout day per course-presentation (in-course only) ---
df_course_dropout = execute_query('''
    SELECT
        code_module,
        code_presentation,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY code_module, code_presentation
''')

# Merge course profile with median dropout day
df_merged = df_course_dropout.merge(
    df_course[['code_module', 'code_presentation',
               'course_length_days', 'assessments_per_30_days',
               'withdrawal_rate_pct']],
    on=['code_module', 'code_presentation'],
)

# --- 1x2 scatter panel: course design vs dropout ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left: assessment density vs withdrawal rate
for _, row in df_merged.iterrows():
    module = row['code_module']
    ax1.scatter(
        row['assessments_per_30_days'], row['withdrawal_rate_pct'],
        color=PALETTE_COURSE.get(module, '#999999'), s=80,
        edgecolor='white', zorder=3,
    )
    ax1.annotate(
        module,
        (row['assessments_per_30_days'], row['withdrawal_rate_pct']),
        fontsize=7, ha='center', va='bottom',
        xytext=(0, 5), textcoords='offset points',
    )
ax1.set_xlabel('Assessments per 30 days')
ax1.set_ylabel('Withdrawal rate (%)')
ax1.set_title('Assessment Density vs. Withdrawal Rate')
sns.despine(ax=ax1)

# Right: course length vs median dropout day
for _, row in df_merged.iterrows():
    module = row['code_module']
    ax2.scatter(
        row['course_length_days'], row['median_dropout_day'],
        color=PALETTE_COURSE.get(module, '#999999'), s=80,
        edgecolor='white', zorder=3,
    )
    ax2.annotate(
        module,
        (row['course_length_days'], row['median_dropout_day']),
        fontsize=7, ha='center', va='bottom',
        xytext=(0, 5), textcoords='offset points',
    )
# Reference line: dropout at course end (y = x diagonal)
max_len = df_merged['course_length_days'].max()
ax2.plot([0, max_len], [0, max_len], color='gray', linestyle='--',
         linewidth=1, alpha=0.5, label='Dropout at course end')
ax2.set_xlabel('Course length (days)')
ax2.set_ylabel(LABEL_MEDIAN_DROPOUT_DAY)
ax2.set_title('Course Length vs. Median Dropout Day')
ax2.legend(fontsize=8)
sns.despine(ax=ax2)

fig.suptitle('Course Design Characteristics and Dropout Patterns',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_course_design_vs_dropout')
plt.show()

> **Interpretation:**
> - **Assessment density** and **withdrawal rate** may show a relationship: courses with more frequent assessments could have different retention profiles. The direction of this relationship (if any) is an empirical question that BQ4 will explore formally.
> - **Course length** and **median dropout day** show that longer courses tend to have later median dropout days in absolute terms, which is expected — but the normalized analysis in Section 5 already showed that the *relative* timing (percentage through course) may be more consistent across modules.
> - The scatter patterns suggest that course design features contribute to — but do not fully explain — the variation in dropout behavior. Student-level factors (demographics, engagement) also play a role, as BQ2 and BQ3 will explore.
>
> **Preview of BQ4:** Notebook 06 will formalize this analysis with per-course retention metrics, assessment-interaction analysis, and resource diversity effects.

## 9. Key Takeaways and Next Steps

### What we learned

1. **Dropout is substantial and universal.** Roughly one-third of all enrollments end in explicit withdrawal. No module is immune, though rates vary significantly across courses.

2. **Cumulative dropout curves reveal distinct temporal profiles.** Some courses experience steep early attrition (onboarding failure), while others show more gradual mid-course decline. The survival-style visualization makes these patterns immediately legible.

3. **Cliff events exist.** Specific days see disproportionately large numbers of withdrawals, likely tied to course milestones (assessment deadlines, grade releases). These are actionable: interventions can be timed to precede known cliff dates.

4. **Normalized timing enables cross-course comparison.** Converting dropout day to percentage-through-course reveals that the *phase* of dropout (early, mid, late) varies by module, suggesting different underlying causes.

5. **Pre-course withdrawals are a distinct phenomenon.** A measurable share of students withdraw before day 0 — pure registration churn. These students need activation, not academic support.

6. **Demographic segmentation shows moderate variation.** Dropout timing differs somewhat by education level, age, and socioeconomic index, but the differences are modest compared to overall variation. Formal testing in BQ3 will determine statistical significance.

7. **Course design features correlate with dropout patterns.** Assessment density and course length show visual associations with dropout rates and timing, but formal analysis in BQ4 is needed to quantify these relationships.

### What comes next

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **04** | BQ2 | Which early behavioral signals predict drop-out? |
| **05** | BQ3 | Demographics vs. behavior — what predicts outcome more? |
| **06** | BQ4 | How do course characteristics affect retention? |
| **07** | BQ5 | Top 3 actionable interventions |

---

**Reproducibility:** All figures are saved to `reports/figures/`. To reproduce this notebook, run `python -m run_pipeline` first, then execute all cells in order.

> **From timing to signals:** This notebook established *when* students leave — the temporal landscape of dropout. But timing alone does not enable prevention. The next step is to identify *what early behaviors predict departure*.
>
> Continue to **Notebook 04** (`04_bq2_early_signals.ipynb`) for BQ2: which early behavioral signals predict drop-out? Notebook 04 introduces formal statistical testing with t-tests, effect sizes, and multiple comparison correction — the first use of `src/stats/tests.py` in analysis.